
# USPS digits benchmark


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter, LogFormatterSciNotation

from src.kernel_pca import KernelPCA, RFF_KernelPCA
from src.usps_benchmarks import *
from src.utils.misc import timing_to_df_rff

# automatic reload
%load_ext autoreload
%autoreload 2


In [ ]:
out_dir = "../../writeup/Kernel_PCA_Preimage_IEEE_TPAMI/"

## Benchmark metrics

In [ ]:
seed = 42
num_repeats = 10

#### effect of number of RFFs

In [ ]:
num_components = 50
num_train = 8000
num_test = 1000
num_rffs = [1000, 2000, 3000, 4000, 5000]


all_metrics_rffs, all_timing_rffs = run_efficiency_benchmark_rff(num_components=num_components, num_train=num_train, num_test=num_test, num_rffs=num_rffs, num_repeats=num_repeats)

In [ ]:
all_metrics_rffs_df = pd.concat(all_metrics_rffs, axis=0)
all_metrics_rffs_df.index = ['MSE', 'SSIM', 'PSNR'] * len(all_metrics_rffs)

all_timing_rffs_df = []
for out_timing in all_timing_rffs:
    all_timing_rffs_df.append(timing_to_df_rff(out_timing))
all_timing_rffs_df = pd.concat(all_timing_rffs_df, axis=0)

# save output to CSV files
all_metrics_rffs_df.to_csv("results/usps_benchmark_efficiency_metrics_num_rffs.csv")
all_timing_rffs_df.to_csv("results/usps_benchmark_efficiency_time_and_memory_num_rffs.csv")

#### effect of number of components

In [ ]:
num_components = [25, 50, 100, 200]
num_train = 6000
num_test = 2000
num_rffs = 2000

all_metrics_comp, all_timing_comp = run_efficiency_benchmark_rff(num_components=num_components, num_train=num_train, num_test=num_test, num_rffs=num_rffs, num_repeats=num_repeats)

In [ ]:
all_metrics_comp_df = pd.concat(all_metrics_comp, axis=0)
all_metrics_comp_df.index = ['MSE', 'SSIM', 'PSNR'] * len(all_metrics_comp)

all_timing_comp_df = []
for out_timing in all_timing_comp:
    all_timing_comp_df.append(timing_to_df_rff(out_timing))
all_timing_comp_df = pd.concat(all_timing_comp_df, axis=0)

# save output to CSV files
all_metrics_comp_df.to_csv("results/usps_benchmark_efficiency_metrics_rff_num_comp.csv")
all_timing_comp_df.to_csv("results/usps_benchmark_efficiency_time_and_memory_rff_num_comp.csv")

#### effect of training data size

In [ ]:
num_components = 50
num_train = [2000, 4000, 6000, 8000]
num_test = 1000
num_rffs = 2000

all_metrics_train, all_timing_train = run_efficiency_benchmark_rff(num_components=num_components, num_train=num_train, num_test=num_test, num_rffs=num_rffs, num_repeats=num_repeats)

In [ ]:
all_metrics_train_df = pd.concat(all_metrics_train, axis=0)
all_metrics_train_df.index = ['MSE', 'SSIM', 'PSNR'] * len(all_metrics_train)

all_timing_train_df = []
for out_timing in all_timing_train:
    all_timing_train_df.append(timing_to_df_rff(out_timing))
all_timing_train_df = pd.concat(all_timing_train_df, axis=0)

# save output to CSV files
all_metrics_train_df.to_csv("results/usps_benchmark_efficiency_metrics_rff_train_size.csv")
all_timing_train_df.to_csv("results/usps_benchmark_efficiency_time_and_memory_rff_train_size.csv")

#### effect of test data size

In [ ]:
num_components = 50
num_train = 4000
num_test = [1000, 2000, 3000, 4000]
num_rffs = 2000

all_metrics_test, all_timing_test = run_efficiency_benchmark_rff(num_components=num_components, num_train=num_train, num_test=num_test, num_rffs=num_rffs, num_repeats=num_repeats)

In [ ]:
all_metrics_test_df = pd.concat(all_metrics_test, axis=0)
all_metrics_test_df.index = ['MSE', 'SSIM', 'PSNR'] * len(all_metrics_test)

all_timing_test_df = []
for out_timing in all_timing_test:
    all_timing_test_df.append(timing_to_df_rff(out_timing))
all_timing_test_df = pd.concat(all_timing_test_df, axis=0)

# save output to CSV files
all_metrics_test_df.to_csv("results/usps_benchmark_efficiency_metrics_rff_test_size.csv")
all_timing_test_df.to_csv("results/usps_benchmark_efficiency_time_and_memory_rff_test_size.csv")

#### effect of kernel bandwidth

In [ ]:
num_components = 50
num_train = 6000
num_test = 2000
num_rffs = 2000
sigma = [5, 10, 20, 40, 80]

all_metrics_sigma, all_timing_sigma = run_efficiency_benchmark_rff(num_components=num_components, num_train=num_train, num_test=num_test, num_rffs=num_rffs, num_repeats=num_repeats, sigma=sigma)

In [ ]:
all_metrics_sigma_df = pd.concat(all_metrics_sigma, axis=0)
all_metrics_sigma_df.index = ['MSE', 'SSIM', 'PSNR'] * len(all_metrics_sigma)

all_timing_sigma_df = []
for out_timing in all_timing_sigma:
    all_timing_sigma_df.append(timing_to_df_rff(out_timing))
all_timing_sigma_df = pd.concat(all_timing_sigma_df, axis=0)

# save output to CSV files
all_metrics_sigma_df.to_csv("results/usps_benchmark_efficiency_metrics_rff_bandwidth.csv")
all_timing_sigma_df.to_csv("results/usps_benchmark_efficiency_time_and_memory_rff_bandwidth.csv")

## Visualization

In [ ]:
def plot_efficiency_benchmark(df, variable, xlabel, out_file=None, logx=True,
     fontsize=8, colors=["black", "red", "blue", "purple", "orange"]):
    """
    Plot the efficiency benchmark results for a given measure.
    """    
    panels = {"MSE": "MSE", "time_s": "Elapsed time (s)", "peak_memory_mb": "Peak memory (MB)"}

    # --- prepare data once ---
    is_mem = df["measure"] == "peak_memory_kb"
    df.loc[is_mem, "value"] /= 1024
    df.loc[is_mem, "measure"] = "peak_memory_mb"

    df["log_value"] = np.log(df["value"])

    stats = (
        df.groupby(["method", "measure"] + [variable])["log_value"]
        .agg(mu="mean", sd="std")
        .reset_index()
        .fillna({"sd": 0})
    )
    stats["center"] = np.exp(stats["mu"])
    stats["lo"] = np.exp(stats["mu"] - stats["sd"])
    stats["hi"] = np.exp(stats["mu"] + stats["sd"])

    # --- plot ---
    #methods = df["method"].unique()
    methods = {'ours': 'Ours', 'bws': 'B.W.S. (RFF approx)', 'tr': 'T.R.'}
    fig, axes = plt.subplots(1, len(panels), figsize=(2*len(panels), 1.5), constrained_layout=True, sharex=True)

    for ax, (measure, ylabel) in zip(axes, panels.items()):
        for method, color in zip(methods.keys(), colors):
            d = stats[(stats["method"] == method) & (stats["measure"] == measure)]
            d = d.sort_values(variable)

            ax.errorbar(d[variable], d["center"], yerr=[d["center"] - d["lo"], d["hi"] - d["center"]],
                marker="o", ms=3.5, lw=1., color=color, capsize=1.5, elinewidth=0.5, label=methods[method])

        ax.set(xscale="log" if logx else "linear", yscale="linear")
        #ax.set_xlabel(xlabel, fontsize=fontsize)
        ax.set_ylabel(ylabel, fontsize=fontsize)
        ax.tick_params(axis="both", which="both", labelsize=fontsize)
        if logx:
            # fewer major ticks (one per decade, or every 2 decades)
            ax.xaxis.set_major_locator(LogLocator(base=10, numticks=5))
            # keep minor tick marks but hide their labels
            ax.xaxis.set_minor_formatter(NullFormatter())
    fig.supxlabel(xlabel, fontsize=fontsize)
    # single shared legend on top
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels, loc="upper center", ncol=len(methods),
        bbox_to_anchor=(0.55, 1.15), fontsize=fontsize, frameon=False)
    if out_file:
        fig.savefig(out_file, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()
    

#### number of testing examples

In [ ]:
metrics_df = all_metrics_test_df.reset_index().rename(columns={'index': 'measure'})
# long format
metrics_df = metrics_df.melt(id_vars=["measure", "n_test", "seed"], 
value_vars=["ours", "bws", "tr"], var_name="method", value_name="value")

timing_df = all_timing_test_df.reset_index().rename(columns={'index': 'measure'})
# replance NA values with 0 for fit_inverse and reconstruction
timing_df['fit_inverse'] = timing_df['fit_inverse'].fillna(0)
timing_df['reconstruction'] = timing_df['reconstruction'].fillna(0)
# total = fit_inverse + reconstruction
timing_df['value'] = timing_df['fit_inverse'] + timing_df['reconstruction']

# metrics & timing
df = pd.concat([timing_df[["measure", "n_test", "seed", "method", "value"]], metrics_df], axis=0) 
outfile = out_dir + "figures/usps_benchmark_efficiency_time_and_memory_test_size_rff.pdf"
plot_efficiency_benchmark(df, 'n_test', 'Number of testing examples', out_file=outfile, logx=False, fontsize=7.5)

#### number of training examples

In [ ]:
metrics_df = all_metrics_train_df.reset_index().rename(columns={'index': 'measure'})
# long format
metrics_df = metrics_df.melt(id_vars=["measure", "n_train", "seed"], 
value_vars=["ours", "bws", "tr"], var_name="method", value_name="value")

timing_df = all_timing_train_df.reset_index().rename(columns={'index': 'measure'})
# replance NA values with 0 for fit_inverse and reconstruction
timing_df['fit_inverse'] = timing_df['fit_inverse'].fillna(0)
timing_df['reconstruction'] = timing_df['reconstruction'].fillna(0)
# total = fit_inverse + reconstruction
timing_df['value'] = timing_df['fit_inverse'] + timing_df['reconstruction']

# metrics & timing
df = pd.concat([timing_df[["measure", "n_train", "seed", "method", "value"]], metrics_df], axis=0) 

outfile = out_dir + "figures/usps_benchmark_efficiency_time_and_memory_train_size_rff.pdf"
plot_efficiency_benchmark(df, 'n_train', 'Number of training examples', out_file=outfile, logx=False, fontsize=7.5)

#### number of retained components

In [ ]:
metrics_df = all_metrics_comp_df.reset_index().rename(columns={'index': 'measure'})
# long format
metrics_df = metrics_df.melt(id_vars=["measure", "n_components", "seed"], 
value_vars=["ours", "bws", "tr"], var_name="method", value_name="value")

timing_df = all_timing_comp_df.reset_index().rename(columns={'index': 'measure'})
# replance NA values with 0 for fit_inverse and reconstruction
timing_df['fit_inverse'] = timing_df['fit_inverse'].fillna(0)
timing_df['reconstruction'] = timing_df['reconstruction'].fillna(0)
# total = fit_inverse + reconstruction
timing_df['value'] = timing_df['fit_inverse'] + timing_df['reconstruction']

# metrics & timing
df = pd.concat([timing_df[["measure", "n_components", "seed", "method", "value"]], metrics_df], axis=0) 

outfile = out_dir + "figures/usps_benchmark_efficiency_time_and_memory_comp_size_rff.pdf"
plot_efficiency_benchmark(df, 'n_components', 'Number of components', out_file=outfile, logx=False, fontsize=7.5)

#### number of RFFs

In [ ]:
metrics_df = all_metrics_rffs_df.reset_index().rename(columns={'index': 'measure'})
# long format
metrics_df = metrics_df.melt(id_vars=["measure", "n_rffs", "seed"], 
value_vars=["ours", "bws", "tr"], var_name="method", value_name="value")

timing_df = all_timing_rffs_df.reset_index().rename(columns={'index': 'measure'})
# replance NA values with 0 for fit_inverse and reconstruction
timing_df['fit_inverse'] = timing_df['fit_inverse'].fillna(0)
timing_df['reconstruction'] = timing_df['reconstruction'].fillna(0)
# total = fit_inverse + reconstruction
timing_df['value'] = timing_df['fit_inverse'] + timing_df['reconstruction']

# metrics & timing
df = pd.concat([timing_df[["measure", "n_rffs", "seed", "method", "value"]], metrics_df], axis=0) 

outfile = out_dir + "figures/usps_benchmark_efficiency_time_and_memory_num_rffs_rff.pdf"
plot_efficiency_benchmark(df, 'n_rffs', 'Number of RFFs', out_file=outfile, logx=False, fontsize=7.5)

#### bandwidth

In [ ]:
metrics_df = all_metrics_sigma_df.reset_index().rename(columns={'index': 'measure'})
# long format
metrics_df = metrics_df.melt(id_vars=["measure", "sigma", "seed"], 
value_vars=["ours", "bws", "tr"], var_name="method", value_name="value")

timing_df = all_timing_sigma_df.reset_index().rename(columns={'index': 'measure'})
# replance NA values with 0 for fit_inverse and reconstruction
timing_df['fit_inverse'] = timing_df['fit_inverse'].fillna(0)
timing_df['reconstruction'] = timing_df['reconstruction'].fillna(0)
# total = fit_inverse + reconstruction
timing_df['value'] = timing_df['fit_inverse'] + timing_df['reconstruction']

# metrics & timing
df = pd.concat([timing_df[["measure", "sigma", "seed", "method", "value"]], metrics_df], axis=0) 

outfile = out_dir + "figures/usps_benchmark_efficiency_time_and_memory_bandwidth_rff.pdf"
plot_efficiency_benchmark(df, 'sigma', 'Kernel bandwidth', out_file=outfile, logx=False, fontsize=7.5)